**Date Functions**

| Function | What it returns | Example |
|----------|-----------------|---------|
| `current_date()` | Returns today's date | `2023-10-15` |
| `current_timestamp()` | Returns the current date and time | `2023-10-15 14:32:05` |
| `year(col)` | Extracts the year as an integer | `2023` |
| `month(col)` | Extracts the month as an integer (`1-12`) | `1` |
| `dayofmonth(col)` | Extracts the day of the month (`1-31`) | `5` |
| `dayofweek(col)` | Extracts the day of the week (`1 = Sunday`, `7 = Saturday`) | `5` |
| `quarter(col)` | Extracts the quarter (`1-4`) | `1` |
| `date_add(col, n)` | Adds `n` days to a date | `2023-01-12` |
| `date_sub(col, n)` | Subtracts `n` days from a date | `2022-12-06` |
| `datediff(end, start)` | Returns the number of days between two dates | `648` |
| `months_between(end, start)` | Returns the number of months between two dates | `21.32` |
| `to_date(col, format)` | Converts a string to `DateType` | `"2023-01-05"` → `DateType` |
| `date_format(col, fmt)` | Formats a date as a string | `"Jan 05, 2023"` |

In [1]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-11")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1d12ff78-a0c3-4fde-a204-3e12b6cdd619;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 141ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

**Task 1**

From orders.csv, extract year, month, day, and quarter from order_date. How many orders were placed in Q1 (quarter = 1)?

In [15]:
from pyspark.sql import functions as F

orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3a://pyspark-30-days-rahul-2026/data/orders.csv")

extract=orders_df.select(
    F.year("order_date").alias("year"),
    F.month("order_date").alias("month"),
    F.dayofmonth("order_date").alias("day"),
    F.quarter("order_date").alias("quarter")
)
extract.show(5)
print(f"total orders placed in Q1: {extract.filter(F.col('quarter')==1).count()}")


+----+-----+---+-------+
|year|month|day|quarter|
+----+-----+---+-------+
|2023|    1|  5|      1|
|2023|    1|  7|      1|
|2023|    1| 10|      1|
|2023|    1| 12|      1|
|2023|    1| 15|      1|
+----+-----+---+-------+
only showing top 5 rows


total orders placed in Q1: 30


**Task 2**

Calculate how many days ago each order was placed using datediff(current_date(), order_date). Show the 5 most recent orders.

In [18]:
orders_df.select(
    "order_date",
    F.datediff(F.current_date(), "order_date").alias("days_since_order")
).show(5)

+----------+----------------+
|order_date|days_since_order|
+----------+----------------+
|2023-01-05|            1301|
|2023-01-07|            1299|
|2023-01-10|            1296|
|2023-01-12|            1294|
|2023-01-15|            1291|
+----------+----------------+
only showing top 5 rows


**Task 3**

Add a delivery_deadline column — 7 days after order_date — using date_add(). Also add a year_month column formatted as yyyy-MM using date_format().

In [24]:
from pyspark.sql import functions as F

orders_df.withColumn(
    "delivery_deadline",
    F.date_add("order_date", 7)
).withColumn(
    "year_month",
    F.date_format("order_date", "yyyy-MM")
).show(5,truncate=False)

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+-----------------+----------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|status   |payment_method|region |delivery_deadline|year_month|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+-----------------+----------+
|O0001   |C001       |P001      |2023-01-05|2       |1299.99   |10          |Delivered|Credit Card   |East   |2023-01-12       |2023-01   |
|O0002   |C002       |P005      |2023-01-07|1       |449.99    |0           |Delivered|PayPal        |West   |2023-01-14       |2023-01   |
|O0003   |C003       |P003      |2023-01-10|4       |349.99    |15          |Delivered|Credit Card   |Midwest|2023-01-17       |2023-01   |
|O0004   |C004       |P006      |2023-01-12|2       |89.99     |5           |Delivered|Debit Card    |South  |2023-01-19       |2023-01   |
|O0005   |C005      

**Task 4**

From customers.csv, use datediff() to calculate how many days each customer has been a member since their signup_date. Show the top 5 longest-standing customers.

In [31]:
from pyspark.sql import functions as F

customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3a://pyspark-30-days-rahul-2026/data/customers.csv")

customers_df.select(
    "customer_id",
    F.datediff(
        F.current_date(),
        "signup_date"
    ).alias("days_since_signup")
).orderBy(
    F.col("days_since_signup").desc()
).show(5)


+-----------+-----------------+
|customer_id|days_since_signup|
+-----------+-----------------+
|       C023|             2283|
|       C009|             2234|
|       C020|             2199|
|       C012|             2187|
|       C006|             2145|
+-----------+-----------------+
only showing top 5 rows
